# GAAE Training — DELCODE Whole-Brain (Schaefer 400)

This notebook trains the GAAE model on whole-brain correlation matrices (400 nodes, Schaefer atlas)
from `DATA/DELCODE/__fc_wholebrain_sch200_flat__/matrices`. It uses `GraphDatasetInMemoryFiltered` for whole-brain data.
It does a subject-level 60/20/20 train/val/test split via pre-generated CSVs.

In [1]:
import os
import sys
import json
from datetime import datetime
from pathlib import Path

import random
import numpy as np
import torch
from torch_geometric.loader import DataLoader

base_dir = Path('/mnt/e/fyassine/ad-early-detection/CLASSIFIER')
if str(base_dir) not in sys.path:
    sys.path.insert(0, str(base_dir))

print(f'Added to sys.path: {base_dir}')

from model.GAAE.models import GraphAttentionAutoencoderConditioned
from model.GAAE.dataset import GraphDatasetInMemoryFiltered
from model.GAAE.utils import knn_binary_adjacency_matrix_no_diag
from model.GAAE.train import train_model_with_val_notebook_train_loss

from common.seeding import set_seed, make_rng, make_torch_generator, seed_worker
from common.provenance import region_from_data_root


Added to sys.path: /mnt/e/fyassine/ad-early-detection/CLASSIFIER


In [2]:
# v2 reproducibility seeding — must run before datasets, samplers, or models.
SEED = 100
set_seed(SEED)
rng = make_rng(SEED)
torch_gen = make_torch_generator(SEED)

## Configuration

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [4]:
WANDB_PROJECT = "ad-early-detection-whole-brain"
try:
    import wandb
    wandb.login()
except Exception:
    wandb = None
    print("wandb not available — logging disabled")

# ── Paths ───────────────────────────────────────────────
WB_DATA_ROOT  = '/mnt/e/fyassine/ad-early-detection/DATA/DELCODE/__fc_wholebrain_sch200_flat__/matrices'
METADATA_DIR  = '/mnt/e/fyassine/ad-early-detection/DATA/DELCODE/__fc_wholebrain_sch200_flat__/metadata'
SPLITS_DIR    = os.path.join(METADATA_DIR, 'splits_gaae')
TRAIN_CSV     = os.path.join(SPLITS_DIR, 'train.csv')
VAL_CSV       = os.path.join(SPLITS_DIR, 'val.csv')
TEST_CSV      = os.path.join(SPLITS_DIR, 'test.csv')

# Brain region / atlas parsed from the DATA directory name. Surfaced in the run
# name and stored in the run config so the input data is visible at a glance.
DATA_INFO = region_from_data_root(WB_DATA_ROOT)
REGION    = DATA_INFO['region']
print(f"Input data: region={DATA_INFO['region']}  atlas={DATA_INFO['atlas']}  ({DATA_INFO['dataset_dir']})")

CHECKPOINT_SEARCH_DIRS = [
    str(base_dir / 'notebooks' / 'checkpoints' / 'checkpoints_gaae_whole_brain'),
]
OUTPUT_DIR = str(base_dir / 'notebooks' / 'checkpoints' / 'checkpoints_gaae_whole_brain')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── GAAE training hyper-params (loaded from configs/, inline fallback) ─────
CONFIG_PATH = base_dir / 'configs' / 'gaae_delcode_whole_brain.json'
TRAIN_CONFIG_PATH = CONFIG_PATH
_train_defaults = {
    "seed": 100, "batch_size": 64, "learning_rate": 0.001, "adj_loss_weight": 0.1,
    "epochs": 1000, "early_stopping_patience": 100, "latent_dim": 64, "num_heads": 2,
    "cond_dim": 2, "dropout": 0.3, "adjacency_k": 8, "num_workers": 8,
    "file_variant": "z_transformed",
}
if TRAIN_CONFIG_PATH.exists():
    with open(TRAIN_CONFIG_PATH) as _f:
        TRAIN_CONFIG = json.load(_f)
    print(f'Loaded training config from {TRAIN_CONFIG_PATH}')
else:
    TRAIN_CONFIG = _train_defaults
    print(f'Training config not found at {TRAIN_CONFIG_PATH} — using inline defaults.')

RANDOM_STATE            = TRAIN_CONFIG['seed']
set_seed(RANDOM_STATE)
BATCH_SIZE              = TRAIN_CONFIG['batch_size']
LEARNING_RATE           = TRAIN_CONFIG['learning_rate']
ADJ_LOSS_WEIGHT         = TRAIN_CONFIG['adj_loss_weight']
EPOCHS                  = TRAIN_CONFIG['epochs']
EARLY_STOPPING_PATIENCE = TRAIN_CONFIG['early_stopping_patience']
LATENT_DIM              = TRAIN_CONFIG['latent_dim']
NUM_HEADS               = TRAIN_CONFIG['num_heads']
COND_DIM                = TRAIN_CONFIG['cond_dim']
DROPOUT                 = TRAIN_CONFIG['dropout']
ADJACENCY_K             = TRAIN_CONFIG['adjacency_k']
ADJACENCY_ARGS          = {'k': ADJACENCY_K}
NUM_WORKERS             = TRAIN_CONFIG['num_workers']
FILE_VARIANT            = TRAIN_CONFIG.get('file_variant', 'z_transformed')

# Load existing checkpoint instead of training?
USE_EXISTING_CHECKPOINT = False

print('Config set.')

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/wunderlich/.netrc.
wandb: Currently logged in as: lakhalfrajyassine (lakhalfrajyassine-technical-university-of-munich) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Input data: region=wholebrain  atlas=sch200  (__fc_wholebrain_sch200_flat__)
Loaded training config from /mnt/e/fyassine/ad-early-detection/CLASSIFIER/configs/gaae_delcode_whole_brain.json
Config set.


## Dataset

In [5]:


train_dataset = GraphDatasetInMemoryFiltered(
    root=WB_DATA_ROOT,
    adjacency_function=knn_binary_adjacency_matrix_no_diag,
    adjacency_args=ADJACENCY_ARGS,
    filter_csv_path=TRAIN_CSV,
    separator=",",
    file_variant=FILE_VARIANT,
)

val_dataset = GraphDatasetInMemoryFiltered(
    root=WB_DATA_ROOT,
    adjacency_function=knn_binary_adjacency_matrix_no_diag,
    adjacency_args=ADJACENCY_ARGS,
    filter_csv_path=VAL_CSV,
    separator=",",
    file_variant=FILE_VARIANT,
)

test_dataset = GraphDatasetInMemoryFiltered(
    root=WB_DATA_ROOT,
    adjacency_function=knn_binary_adjacency_matrix_no_diag,
    adjacency_args=ADJACENCY_ARGS,
    filter_csv_path=TEST_CSV,
    separator=",",
    file_variant=FILE_VARIANT,
)

print(f"Train: {len(train_dataset)} samples")
print(f"Val:   {len(val_dataset)} samples")
print(f"Test:  {len(test_dataset)} samples")
print(f"Feature shape: {train_dataset[0].x.shape}")

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    worker_init_fn=seed_worker,
    generator=torch_gen,
    persistent_workers=True if NUM_WORKERS > 0 else False,
    pin_memory=True if torch.cuda.is_available() else False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    worker_init_fn=seed_worker,
    generator=torch_gen,
    persistent_workers=True if NUM_WORKERS > 0 else False,
    pin_memory=True if torch.cuda.is_available() else False
)

dataset_info = {
    "dataset_name": "Whole-brain Schaefer 400-node (60/20/20 subject-level split)",
    "kNN_param": ADJACENCY_ARGS['k'],
    "correlation_type": FILE_VARIANT,
    "num_features": train_dataset[0].x.size(1),
    "train_dataset_size": len(train_dataset),
    "val_dataset_size": len(val_dataset),
    "test_dataset_size": len(test_dataset),
    "batch_size": BATCH_SIZE
}
print(f"Dataset info: {json.dumps(dataset_info, indent=2)}")

Train: 582 samples
Val:   113 samples
Test:  78 samples
Feature shape: torch.Size([200, 200])
Dataset info: {
  "dataset_name": "Whole-brain Schaefer 400-node (60/20/20 subject-level split)",
  "kNN_param": 8,
  "correlation_type": "z_transformed",
  "num_features": 200,
  "train_dataset_size": 582,
  "val_dataset_size": 113,
  "test_dataset_size": 78,
  "batch_size": 64
}


## Model

In [6]:
in_features = train_dataset[0].x.size(1)
hidden_dim = in_features

In [7]:
model = GraphAttentionAutoencoderConditioned(
    in_features=in_features,
    hidden_dim=hidden_dim,
    out_features=LATENT_DIM,
    cond_dim=COND_DIM,
    num_heads=NUM_HEADS,
    dropout=DROPOUT
).to(device)

model_config = {
    "model_type": model.__class__.__name__,
    "in_features": in_features,
    "hidden_size": hidden_dim,
    "latent_dim": LATENT_DIM,
    "attention_heads": NUM_HEADS,
    "device": device.type,
    "dropout": DROPOUT
}

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
print(f"Model: {model.__class__.__name__}")
print(f"  in_features={in_features}, hidden_dim={hidden_dim}, out_features={LATENT_DIM}")
print(f"  num_heads={NUM_HEADS}, dropout={DROPOUT}")

Model: GraphAttentionAutoencoderConditioned
  in_features=200, hidden_dim=200, out_features=64
  num_heads=2, dropout=0.3


In [ ]:
CHECKPOINT_SEARCH_DIRS = [
    str(base_dir / "notebooks" / "checkpoints" / "checkpoints_gaae_whole_brain"),
]

if USE_EXISTING_CHECKPOINT:
    checkpoint_candidates = sorted(
        [
            (run_dir.name, str(model_file), str(run_dir))
            for checkpoint_dir in CHECKPOINT_SEARCH_DIRS
            for base_dir_path in [Path(checkpoint_dir)]
            if base_dir_path.is_dir()
            for run_dir in sorted(base_dir_path.iterdir())
            if run_dir.is_dir()
            for model_file in [run_dir / f"model_{run_dir.name}.pth"]
            if model_file.exists()
        ],
        key=lambda x: x[0],
    )

    if not checkpoint_candidates:
        raise FileNotFoundError(
            "No GAAE checkpoints found. Train first or set USE_EXISTING_CHECKPOINT=False."
        )

    print("Available GAAE checkpoints:")
    for i, (run_name, _, run_dir) in enumerate(checkpoint_candidates):
        print(f"  {i}: {run_name} ({run_dir})")

    selected_run_index = int(input("Select a checkpoint by index: "))
    SELECTED_RUN_NAME, GAAE_CHECKPOINT_PATH, SELECTED_RUN_DIR = checkpoint_candidates[selected_run_index]

    checkpoint_obj = torch.load(GAAE_CHECKPOINT_PATH, map_location=device)
    if isinstance(checkpoint_obj, torch.nn.Module):
        model = checkpoint_obj.to(device)
    elif isinstance(checkpoint_obj, dict):
        model.load_state_dict(checkpoint_obj)
        model = model.to(device)
    else:
        raise TypeError("Unsupported checkpoint format. Expected torch.nn.Module or state_dict.")

    best_model = model
    history = {"train_loss": [], "val_loss": []}
    run_timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    run_name = f"checkpoint_{SELECTED_RUN_NAME}_{run_timestamp}"
    print(f"Loaded checkpoint from {GAAE_CHECKPOINT_PATH}")
    print("Training skipped because USE_EXISTING_CHECKPOINT=True")
else:
    best_model, history = train_model_with_val_notebook_train_loss(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=optimizer,
        device=device,
        batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        model_config=model_config,
        adj_loss_weight=ADJ_LOSS_WEIGHT,
        epochs=EPOCHS,
        early_stopping_patience=EARLY_STOPPING_PATIENCE,
        dataset_info=dataset_info,
        project_name=WANDB_PROJECT
    )

    run_timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    try:
        wandb_run_name = wandb.run.name if wandb and wandb.run and wandb.run.name else "run"
    except Exception:
        wandb_run_name = "run"
    wandb_run_name = str(wandb_run_name).replace(" ", "-")
    run_name = f"{wandb_run_name}_{run_timestamp}"

    checkpoint_root = base_dir / "notebooks" / "checkpoints" / "checkpoints_gaae_whole_brain"
    os.makedirs(checkpoint_root, exist_ok=True)
    run_artifact_dir = checkpoint_root / run_name
    os.makedirs(run_artifact_dir, exist_ok=True)

    model_file = run_artifact_dir / f"model_{run_name}.pth"
    torch.save(best_model, str(model_file))
    print(f"Saved best model to {model_file}")

    config_to_save = {
        "run_name": run_name,
        "wandb_run_name": wandb_run_name,
        "timestamp": run_timestamp,
        "dataset_info": dataset_info,
        "model_config": model_config,
        "training_config": {
            "batch_size": BATCH_SIZE,
            "learning_rate": LEARNING_RATE,
            "adj_loss_weight": ADJ_LOSS_WEIGHT,
            "epochs": EPOCHS,
            "early_stopping_patience": EARLY_STOPPING_PATIENCE
        }
    }

    def json_serial(obj):
        if isinstance(obj, (datetime, torch.device)):
            return str(obj)
        raise TypeError(f"Type {type(obj)} not serializable")

    config_file = run_artifact_dir / "run_config.json"
    with open(config_file, "w") as f:
        json.dump(config_to_save, f, indent=4, default=json_serial)
    print(f"Saved run configuration to {config_file}")

Training Progress:   0%|          | 0/1000 [00:00<?, ?it/s]

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Validation Loss')
plt.title('Training and Validation Loss (Whole-Brain 400-node)')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# state = torch.load(str("../Fabian's models/200_Delcode_healthy_age_sex_conditioned/best_model_4000_epochs.pth"), map_location=device)
# model.load_state_dict(state)
# model.to(device)

## Loss

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import torch.nn.functional as F
from torch_geometric.utils import to_dense_adj

from model.GAAE.utils import calculate_dense_adjacency

train_split_df = pd.read_csv(TRAIN_CSV)
val_split_df = pd.read_csv(VAL_CSV)

assert "Repseudonym" in train_split_df.columns and "diagnosis" in train_split_df.columns
assert "Repseudonym" in val_split_df.columns and "diagnosis" in val_split_df.columns

allowed_cohorts = {"ad", "converter", "healthy", "mci"}

combined_split_df = pd.concat([train_split_df, val_split_df], ignore_index=True)
subject_cohort_map = (
    combined_split_df[["Repseudonym", "diagnosis"]]
    .drop_duplicates(subset=["Repseudonym"], keep="first")
    .assign(
        Repseudonym=lambda d: d["Repseudonym"].astype(str).str.strip(),
        diagnosis=lambda d: d["diagnosis"].astype(str).str.lower().str.strip(),
    )
    .set_index("Repseudonym")["diagnosis"]
    .to_dict()
)

def compute_split_errors_by_cohort(dataset, split_name):
    records = []
    unknown_ids = []
    model.eval()

    for data in dataset:
        data = data.to(device)
        x, edge_index, edge_attr = data.x, data.edge_index, getattr(data, "edge_attr", None)
        patient_id = str(getattr(data, "patient_id", "")).strip()
        cohort = subject_cohort_map.get(patient_id)

        if cohort not in allowed_cohorts:
            unknown_ids.append(patient_id)
            continue

        cond_vec = torch.tensor(
            [[data.patient_age.item(), float(data.patient_sex.item())]],
            device=device,
        )
        batch_mask = torch.zeros(x.size(0), dtype=torch.long, device=device)

        with torch.no_grad():
            _, x_reconstructed, adj_reconstructed, _ = model(
                x, edge_index, edge_attr, cond_vec, batch_mask
            )

        x_error = F.mse_loss(x_reconstructed, x).item()
        adj_original = calculate_dense_adjacency(data).to(device)
        adj_reconstructed_dense = to_dense_adj(
            edge_index, edge_attr=adj_reconstructed, max_num_nodes=x.size(0)
        ).squeeze(0)
        adj_error = F.binary_cross_entropy(adj_reconstructed_dense, adj_original).item()
        total_error = x_error + ADJ_LOSS_WEIGHT * adj_error

        records.append(
            {
                "Split": split_name,
                "Cohort": cohort,
                "Total Error": total_error,
            }
        )

    if unknown_ids:
        unique_unknown = sorted(set(unknown_ids))
        raise ValueError(
            f"{split_name}: found {len(unique_unknown)} IDs without allowed diagnosis mapping. "
            f"Examples: {unique_unknown[:10]}"
        )

    return pd.DataFrame(records)

train_errors_df = compute_split_errors_by_cohort(train_dataset, "Train")
val_errors_df = compute_split_errors_by_cohort(val_dataset, "Validation")

cohort_order = [
    c for c in ["ad", "converter", "healthy", "mci"]
    if c in set(train_errors_df["Cohort"]) | set(val_errors_df["Cohort"])
]

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

split_frames = [
    ("Train", train_errors_df, "Blues"),
    ("Validation", val_errors_df, "Greens"),
]

for split_name, split_df, palette_name in split_frames:
    plt.figure(figsize=(10, 7))
    ax = plt.gca()

    sns.swarmplot(
        data=split_df,
        x="Cohort",
        y="Total Error",
        order=cohort_order,
        color=".25",
        size=4,
        alpha=0.6,
        zorder=1,
        ax=ax,
    )

    boxplot = sns.boxplot(
        data=split_df,
        x="Cohort",
        y="Total Error",
        order=cohort_order,
        palette=palette_name,
        showcaps=True,
        boxprops={"edgecolor": "black", "linewidth": 2},
        medianprops={"color": "red", "linewidth": 2.5},
        whiskerprops={"color": "black", "linewidth": 2},
        capprops={"color": "black", "linewidth": 2},
        zorder=2,
        ax=ax,
    )

    alpha_value = 0.5
    for patch in boxplot.patches:
        facecolor = patch.get_facecolor()
        if len(facecolor) == 4:
            r, g, b, _ = facecolor
        else:
            r, g, b = facecolor[:3]
        patch.set_facecolor((r, g, b, alpha_value))

    y_top = split_df["Total Error"].max()
    y_min = split_df["Total Error"].min()
    y_span = y_top - y_min if y_top > y_min else 1.0
    ax.set_ylim(y_min - 0.05 * y_span, y_top + 0.18 * y_span)

    for i, cohort in enumerate(cohort_order):
        cohort_data = split_df[split_df["Cohort"] == cohort]["Total Error"]
        if cohort_data.empty:
            continue
        mean_val = cohort_data.mean()
        std_val = cohort_data.std(ddof=1) if len(cohort_data) > 1 else 0.0
        ax.text(
            i,
            y_top + 0.12 * y_span,
            f"μ={mean_val:.4f}\nσ={std_val:.4f}",
            ha="center",
            va="top",
            fontsize=9,
            fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.4", facecolor="white", edgecolor="gray", alpha=0.8),
        )

    ax.set_title(f"{split_name} Cohorts", fontsize=16, fontweight="bold")
    ax.set_xlabel("Cohort", fontsize=14)
    ax.set_ylabel("Total Weighted Reconstruction Error", fontsize=14)
    ax.tick_params(axis="x", rotation=25, labelsize=12)
    ax.tick_params(axis="y", labelsize=12)
    ax.grid(axis="y", alpha=0.3, linestyle="--")

    if globals().get("USE_EXISTING_CHECKPOINT", False) and "SELECTED_RUN_NAME" in globals():
        display_run_name = SELECTED_RUN_NAME
    else:
        display_run_name = globals().get("run_name", "run")

    plt.figtext(
        0.99,
        0.01,
        f"Project: {WANDB_PROJECT}, Run: {display_run_name}",
        horizontalalignment="right",
        verticalalignment="bottom",
        fontsize=6,
        alpha=0.5,
    )

    plt.tight_layout(rect=[0, 0.03, 1, 1])
    plt.show()

print("Train cohort counts:")
print(train_errors_df.groupby("Cohort").size().sort_index())

print("\nValidation cohort counts:")
print(val_errors_df.groupby("Cohort").size().sort_index())

In [ ]:
# reconstruction_errors = {
#     'Train': {
#         'x_errors': train_x_errors,
#         'adj_errors': train_adj_errors,
#         'total_errors': train_total_weighted_errors,
#         'patient_ids': train_ids
#     },
#     'Val': {
#         'x_errors': val_x_errors,
#         'adj_errors': val_adj_errors,
#         'total_errors': val_total_weighted_errors,
#         'patient_ids': val_ids
#     }
# }

# for key in reconstruction_errors.keys():
#     x_errors = reconstruction_errors[key]['x_errors']
#     adj_errors = reconstruction_errors[key]['adj_errors']
#     total_errors = reconstruction_errors[key]['total_errors']
#     patient_ids = reconstruction_errors[key]['patient_ids']

#     total_errors = np.array(total_errors)
#     if total_errors.size == 0:
#         reconstruction_errors[key]['x_errors'] = []
#         reconstruction_errors[key]['adj_errors'] = []
#         reconstruction_errors[key]['total_errors'] = []
#         reconstruction_errors[key]['patient_ids'] = []
#         continue

#     valid_mask = ~np.isnan(total_errors)

#     reconstruction_errors[key]['x_errors'] = np.array(x_errors)[valid_mask].tolist()
#     reconstruction_errors[key]['adj_errors'] = np.array(adj_errors)[valid_mask].tolist()
#     reconstruction_errors[key]['total_errors'] = total_errors[valid_mask].tolist()
#     reconstruction_errors[key]['patient_ids'] = np.array(patient_ids)[valid_mask].tolist()

### Train vs. Validation

In [ ]:
# from sklearn.metrics import roc_curve, auc, confusion_matrix, classification_report, precision_recall_curve

# train_errors = reconstruction_errors['Train']['total_errors']
# val_errors = reconstruction_errors['Val']['total_errors']

# all_errors = train_errors + val_errors

# print(f"Total samples: {len(all_errors)}")
# print(f"  Train: {len(train_errors)}")
# print(f"  Val: {len(val_errors)}")

In [ ]:
# data = []
# for dataset_name, errors in reconstruction_errors.items():
#     total_errors = errors['total_errors']
#     for error in total_errors:
#         data.append({'Dataset': dataset_name, 'Total Error': error})

# df = pd.DataFrame(data)
# df = df.dropna(subset=['Total Error'])

# label_map = {
#     'Train': 'Train (All Cohorts)',
#     'Val': 'Val (All Cohorts)'
# }
# df['Cohort'] = df['Dataset'].map(label_map)
# df = df.dropna(subset=['Cohort'])
# cohort_order = [
#     'Train (All Cohorts)',
#     'Val (All Cohorts)'
# ]

# print(f"Dataset sizes:")
# for cohort in cohort_order:
#     count = len(df[df['Cohort'] == cohort])
#     print(f"  {cohort}: {count}")

In [ ]:
# plt.rcParams['pdf.fonttype'] = 42
# plt.rcParams['ps.fonttype'] = 42

# plt.figure(figsize=(10, 7))
# sns.swarmplot(
#     x='Cohort', y='Total Error', data=df,
#     order=cohort_order,
#     color=".25", size=4, alpha=0.6, zorder=1
# )

# boxplot = sns.boxplot(
#     x='Cohort', y='Total Error', data=df,
#     order=cohort_order,
#     palette='Blues',
#     showcaps=True,
#     boxprops={'edgecolor': 'black', 'linewidth': 2},
#     medianprops={'color': 'red', 'linewidth': 2.5},
#     whiskerprops={'color': 'black', 'linewidth': 2},
#     capprops={'color': 'black', 'linewidth': 2},
#     zorder=2
# )

# alpha_value = 0.5
# for patch in boxplot.patches:
#     facecolor = patch.get_facecolor()
#     if len(facecolor) == 4:
#         r, g, b, a = facecolor
#     else:
#         r, g, b = facecolor[:3]
#         a = 1.0
#     patch.set_facecolor((r, g, b, alpha_value))

# for i, cohort in enumerate(cohort_order):
#     cohort_data = df[df['Cohort'] == cohort]['Total Error']
#     mean_val = cohort_data.mean()
#     median_val = cohort_data.median()
#     plt.text(i, plt.ylim()[1] * 0.98, f'μ={mean_val:.4f}\nM={median_val:.4f}', 
#              ha='center', va='top', fontsize=9, fontweight='bold',
#              bbox=dict(boxstyle='round,pad=0.4', facecolor='white', edgecolor='gray', alpha=0.8))

# plt.title('Reconstruction Errors\nTrained on: AD + MCI Stable + Healthy + Converter (85/15)', fontsize=16, fontweight='bold')
# plt.xlabel('Cohort', fontsize=14)
# plt.ylabel('Total Weighted Reconstruction Error', fontsize=14)
# plt.xticks(rotation=0, fontsize=12)
# plt.yticks(fontsize=12)
# plt.grid(axis='y', alpha=0.3, linestyle='--')

# plt.figtext(0.99, 0.01, f"Project: {WANDB_PROJECT}, Run: {wandb_run_name}", horizontalalignment='right', verticalalignment='bottom', fontsize=6, alpha=0.5)
# plt.tight_layout()
# plt.show()

### Healthy vs. AD vs. Converters vs. sMCI Boxplots

In [ ]:
# val_indices_in_subset = val_dataset.indices
# val_indices_in_original = [training_indices[i] for i in val_indices_in_subset]

# val_cohorts = {
#     'AD': [],
#     'MCI Stable': [],
#     'Healthy': [],
#     'Converter': []
# }

# for subset_idx, original_idx in zip(val_indices_in_subset, val_indices_in_original):
#     for cohort_name, (start, end) in cohort_offsets.items():
#         if start <= original_idx < end:
#             val_cohorts[cohort_name].append(subset_idx)
#             break

# print(f"Validation cohort sizes:")
# for cohort, indices in val_cohorts.items():
#     print(f"  {cohort}: {len(indices)}")

In [ ]:
# from torch.utils.data import Subset

# val_cohort_errors = {}

# for cohort_name, cohort_indices in val_cohorts.items():
#     cohort_subset = Subset(combined_dataset, cohort_indices)
    
#     x_errors, adj_errors, total_errors, patient_ids = \
#         evaluate_reconstruction_errors_with_ids(cohort_subset, model, device, adj_loss_weight=ADJ_LOSS_WEIGHT)
    
#     total_errors = np.array(total_errors)
#     valid_mask = ~np.isnan(total_errors)
    
#     val_cohort_errors[cohort_name] = {
#         'x_errors': np.array(x_errors)[valid_mask].tolist(),
#         'adj_errors': np.array(adj_errors)[valid_mask].tolist(),
#         'total_errors': total_errors[valid_mask].tolist(),
#         'patient_ids': np.array(patient_ids)[valid_mask].tolist()
#     }
    
#     print(f"{cohort_name}: {len(total_errors[valid_mask])} valid samples")

In [ ]:
# val_data = []
# for cohort_name, errors in val_cohort_errors.items():
#     total_errors = errors['total_errors']
#     for error in total_errors:
#         val_data.append({'Cohort': cohort_name, 'Total Error': error})

# val_df = pd.DataFrame(val_data)
# val_df = val_df.dropna(subset=['Total Error'])

# val_cohort_order = ['Healthy', 'MCI Stable', 'Converter', 'AD']

# print(f"Validation cohort sizes:")
# for cohort in val_cohort_order:
#     count = len(val_df[val_df['Cohort'] == cohort])
#     print(f"  {cohort}: {count}")

In [ ]:
# plt.rcParams['pdf.fonttype'] = 42
# plt.rcParams['ps.fonttype'] = 42

# plt.figure(figsize=(14, 7))
# sns.swarmplot(
#     x='Cohort', y='Total Error', data=val_df,
#     order=val_cohort_order,
#     color=".25", size=4, alpha=0.6, zorder=1
# )

# boxplot = sns.boxplot(
#     x='Cohort', y='Total Error', data=val_df,
#     order=val_cohort_order,
#     palette='Blues',
#     showcaps=True,
#     boxprops={'edgecolor': 'black', 'linewidth': 2},
#     medianprops={'color': 'red', 'linewidth': 2.5},
#     whiskerprops={'color': 'black', 'linewidth': 2},
#     capprops={'color': 'black', 'linewidth': 2},
#     zorder=2
# )

# alpha_value = 0.5
# for patch in boxplot.patches:
#     r, g, b, _ = patch.get_facecolor()
#     patch.set_facecolor((r, g, b, alpha_value))

# for i, cohort in enumerate(val_cohort_order):
#     cohort_data = val_df[val_df['Cohort'] == cohort]['Total Error']
#     mean_val = cohort_data.mean()
#     median_val = cohort_data.median()
#     plt.text(i, plt.ylim()[1] * 0.98, f'μ={mean_val:.4f}\nM={median_val:.4f}', 
#              ha='center', va='top', fontsize=9, fontweight='bold',
#              bbox=dict(boxstyle='round,pad=0.4', facecolor='white', edgecolor='gray', alpha=0.8))

# plt.title('Validation Reconstruction Errors by Cohort\nTrained on: AD + MCI Stable + Healthy + Converter (85/15)', fontsize=16, fontweight='bold')
# plt.xlabel('Cohort', fontsize=14)
# plt.ylabel('Total Weighted Reconstruction Error', fontsize=14)
# plt.xticks(rotation=25, fontsize=12)
# plt.yticks(fontsize=12)
# plt.grid(axis='y', alpha=0.3, linestyle='--')

# plt.figtext(0.99, 0.01, f"Project: {WANDB_PROJECT}, Run: {wandb_run_name}", horizontalalignment='right', verticalalignment='bottom', fontsize=6, alpha=0.5)
# plt.tight_layout()
# plt.show()

In [ ]:
# summary_table = df.groupby('Cohort')['Total Error'].agg(['count', 'mean', 'std', 'min', 'max']).reindex(cohort_order)

# if 'run_artifact_dir' not in globals():
#     run_artifact_dir = os.path.join("checkpoints", wandb_run_name)
#     os.makedirs(run_artifact_dir, exist_ok=True)

# loss_artifact = {
#     'timestamp': datetime.now().strftime("%Y-%m-%d_%H-%M-%S"),
#     'run_name': wandb_run_name,
#     'model_path': model_file if 'model_file' in globals() else None,
#     'dataset_info': dataset_info,
#     'reconstruction_errors': reconstruction_errors,
#     'cohort_dataframe': df.to_dict(orient="list"),
#     'summary_table': summary_table.reset_index().to_dict(orient="list"),
#     'cohort_order': cohort_order,
#     'label_map': label_map
# }

# loss_filename = f"losses_{wandb_run_name}.pkl"
# loss_file = os.path.join(run_artifact_dir, loss_filename)
# with open(loss_file, "wb") as fp:
#     pickle.dump(loss_artifact, fp)

# print(f"Saved loss artifacts to {loss_file}")

In [ ]:
try:
    wandb.finish()
except Exception:
    pass

## Robustness Evaluation (Threshold from Reconstruction Error)

In [ ]:
import json
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
from sklearn.metrics import roc_curve, roc_auc_score
from model.GAAE.utils import calculate_dense_adjacency

np.random.seed(42)
torch.manual_seed(42)

if 'best_model' in globals() and best_model is not None:
    eval_model = best_model
elif 'model' in globals() and model is not None:
    eval_model = model
else:
    raise RuntimeError('No trained model found. Run training/checkpoint loading cells first.')

eval_model = eval_model.to(device)
eval_model.eval()

if 'val_errors_df' not in globals():
    raise RuntimeError('val_errors_df is missing. Run the cohort reconstruction error cell first.')
if 'test_dataset' not in globals():
    raise RuntimeError('test_dataset is missing. Run dataset loading cells first.')
if 'ADJACENCY_ARGS' not in globals():
    raise RuntimeError('ADJACENCY_ARGS is missing. Run configuration cells first.')

if 'subject_cohort_map' in globals() and isinstance(subject_cohort_map, dict):
    subject_cohort_map_all = dict(subject_cohort_map)
else:
    subject_cohort_map_all = {}
if 'TEST_CSV' in globals():
    test_split_df = pd.read_csv(TEST_CSV)
    test_map = (
        test_split_df[['Repseudonym', 'diagnosis']]
        .dropna()
        .drop_duplicates(subset=['Repseudonym'], keep='first')
        .assign(
            Repseudonym=lambda d: d['Repseudonym'].astype(str),
            diagnosis=lambda d: d['diagnosis'].astype(str).str.lower().str.strip(),
        )
        .set_index('Repseudonym')['diagnosis']
        .to_dict()
    )
    subject_cohort_map_all.update(test_map)

def clone_sample(sample):
    cloned = sample.clone()
    if hasattr(cloned, 'x') and torch.is_tensor(cloned.x):
        cloned.x = cloned.x.clone()
    if hasattr(cloned, 'edge_index') and torch.is_tensor(cloned.edge_index):
        cloned.edge_index = cloned.edge_index.clone()
    if hasattr(cloned, 'edge_attr') and torch.is_tensor(cloned.edge_attr):
        cloned.edge_attr = cloned.edge_attr.clone()
    return cloned

def infer_total_error(sample, noise_method='none', noise_level=0.0):
    d = clone_sample(sample)
    if noise_method in {'feature_noise', 'matrix_noise_rebuild'} and noise_level > 0:
        base_std = float(d.x.std().item()) if d.x.numel() > 1 else 1.0
        d.x = d.x + torch.randn_like(d.x) * (base_std * noise_level)
    if noise_method == 'matrix_noise_rebuild':
        adj_bin = knn_binary_adjacency_matrix_no_diag(d.x.detach().cpu().numpy(), **ADJACENCY_ARGS)
        src, dst = np.where(adj_bin > 0)
        if len(src) == 0:
            src = np.arange(max(0, d.x.size(0) - 1))
            dst = src + 1
        edge_index = np.vstack([src, dst])
        d.edge_index = torch.tensor(edge_index, dtype=torch.long)
        if hasattr(d, 'edge_attr'):
            d.edge_attr = torch.ones(d.edge_index.size(1), dtype=torch.float32)
    if noise_method == 'edge_perturbation' and noise_level > 0:
        edge_index = d.edge_index.detach().cpu()
        num_edges = edge_index.size(1)
        keep = max(1, int(round(num_edges * (1.0 - noise_level))))
        keep_perm = torch.randperm(num_edges)[:keep]
        kept_edges = edge_index[:, keep_perm]
        add_count = int(round(num_edges * noise_level))
        n_nodes = d.x.size(0)
        if add_count > 0:
            new_src = torch.randint(0, n_nodes, (add_count,), dtype=torch.long)
            new_dst = torch.randint(0, n_nodes, (add_count,), dtype=torch.long)
            valid = new_src != new_dst
            new_src = new_src[valid]
            new_dst = new_dst[valid]
            if new_src.numel() > 0:
                add_edges = torch.stack([new_src, new_dst], dim=0)
                d.edge_index = torch.cat([kept_edges, add_edges], dim=1)
            else:
                d.edge_index = kept_edges
        else:
            d.edge_index = kept_edges
        if hasattr(d, 'edge_attr'):
            d.edge_attr = torch.ones(d.edge_index.size(1), dtype=torch.float32)

    age = float(d.patient_age.item()) if torch.is_tensor(d.patient_age) else float(d.patient_age)
    sex = float(d.patient_sex.item()) if torch.is_tensor(d.patient_sex) else float(d.patient_sex)
    if noise_method == 'conditioning_noise' and noise_level > 0:
        age = age + float(np.random.normal(0.0, noise_level * 0.05))
        sex = float(np.clip(sex + np.random.normal(0.0, noise_level * 0.1), 0.0, 1.0))

    d = d.to(device)
    batch_mask = torch.zeros(d.x.size(0), dtype=torch.long, device=device)
    cond_vec = torch.tensor([[age, sex]], dtype=torch.float32, device=device)
    edge_attr = d.edge_attr if hasattr(d, 'edge_attr') else None

    with torch.no_grad():
        _, x_reconstructed, adj_reconstructed, _ = eval_model(d.x, d.edge_index, edge_attr, cond_vec, batch_mask)

    x_error = F.mse_loss(x_reconstructed, d.x).item()
    try:
        adj_true = calculate_dense_adjacency(d).to(device)
        adj_pred = adj_reconstructed[0] if hasattr(adj_reconstructed, 'dim') and adj_reconstructed.dim() == 3 else adj_reconstructed
        adj_error = F.binary_cross_entropy(adj_pred, adj_true).item()
    except Exception:
        adj_error = 0.0
    total_error = x_error + float(ADJ_LOSS_WEIGHT) * adj_error
    return x_error, adj_error, total_error

def compute_errors_for_dataset_with_map(dataset, split_name, cohort_map):
    rows = []
    for i in range(len(dataset)):
        sample = dataset[i]
        patient_id = str(getattr(sample, 'patient_id', f'idx_{i}'))
        cohort = str(cohort_map.get(patient_id, 'unknown')).lower()
        x_err, adj_err, total_err = infer_total_error(sample, noise_method='none', noise_level=0.0)
        rows.append(
            {
                'Split': split_name,
                'DatasetIndex': i,
                'PatientID': patient_id,
                'Cohort': cohort,
                'X Error': float(x_err),
                'Adj Error': float(adj_err),
                'Total Error': float(total_err),
            }
        )
    return pd.DataFrame(rows)

def compute_one_vs_rest_thresholds_from_val(val_errors_df_in, cohorts):
    out = {}
    val_local = val_errors_df_in.copy()
    val_local['Cohort'] = val_local['Cohort'].astype(str).str.lower().str.strip()
    val_errors = val_local['Total Error'].astype(float).values

    for cohort_name in cohorts:
        y = (val_local['Cohort'] == cohort_name).astype(int).values
        if y.sum() == 0 or y.sum() == len(y):
            out[cohort_name] = {
                'direction': 'low',
                'auc': float('nan'),
                'threshold_error': float('nan'),
                'threshold_score': float('nan'),
            }
            continue

        scores_high = val_errors
        scores_low = -val_errors
        auc_high = roc_auc_score(y, scores_high)
        auc_low = roc_auc_score(y, scores_low)

        if auc_high >= auc_low:
            direction = 'high'
            scores = scores_high
            auc_used = auc_high
        else:
            direction = 'low'
            scores = scores_low
            auc_used = auc_low

        fpr, tpr, score_thresholds = roc_curve(y, scores)
        best_idx = int(np.argmax(tpr - fpr))
        threshold_score = float(score_thresholds[best_idx])
        threshold_error = float(threshold_score if direction == 'high' else -threshold_score)

        out[cohort_name] = {
            'direction': direction,
            'auc': float(auc_used),
            'threshold_error': threshold_error,
            'threshold_score': threshold_score,
        }
    return out

def cohort_positive_prediction(total_error, cohort_threshold):
    direction = cohort_threshold['direction']
    threshold_error = cohort_threshold['threshold_error']
    if np.isnan(threshold_error):
        return 0
    if direction == 'high':
        return int(total_error >= threshold_error)
    return int(total_error <= threshold_error)

cohorts_to_analyze = ['healthy', 'ad', 'mci', 'converter']
cohort_thresholds = compute_one_vs_rest_thresholds_from_val(val_errors_df, cohorts_to_analyze)

print('Validation one-vs-rest thresholds by cohort:')
threshold_rows = []
for c in cohorts_to_analyze:
    threshold_rows.append(
        {
            'Cohort': c,
            'Direction': cohort_thresholds[c]['direction'],
            'AUC': cohort_thresholds[c]['auc'],
            'ThresholdError': cohort_thresholds[c]['threshold_error'],
        }
    )
print(pd.DataFrame(threshold_rows).to_string(index=False))

test_errors_eval_df = compute_errors_for_dataset_with_map(test_dataset, 'Test', subject_cohort_map_all)
for c in cohorts_to_analyze:
    pred_col = f'Pred_{c}'
    test_errors_eval_df[pred_col] = test_errors_eval_df['Total Error'].apply(
        lambda e: cohort_positive_prediction(float(e), cohort_thresholds[c])
    )

top_k = 5
selected_frames = []
for cohort_name in cohorts_to_analyze:
    pred_col = f'Pred_{cohort_name}'
    th = cohort_thresholds[cohort_name]
    if np.isnan(th['threshold_error']):
        print(f'Skipping cohort={cohort_name} because threshold is NaN.')
        continue

    cohort_candidates = test_errors_eval_df[
        (test_errors_eval_df['Cohort'] == cohort_name) &
        (test_errors_eval_df[pred_col] == 1)
    ].copy()

    if th['direction'] == 'high':
        cohort_candidates['SelectionMargin'] = cohort_candidates['Total Error'] - th['threshold_error']
    else:
        cohort_candidates['SelectionMargin'] = th['threshold_error'] - cohort_candidates['Total Error']

    if cohort_candidates.empty:
        print(f'No eligible samples found for cohort={cohort_name} with cohort-specific threshold.')
        continue

    selected = cohort_candidates.sort_values('SelectionMargin', ascending=False).head(top_k).copy()
    selected['SelectionCohort'] = cohort_name
    selected['SelectionRule'] = 'correct_by_cohort_threshold'
    selected_frames.append(selected)

if not selected_frames:
    raise RuntimeError('No selected samples found for any requested cohort.')

selected_samples_df = pd.concat(selected_frames, ignore_index=True)
print('')
print('Selected samples per cohort:')
print(selected_samples_df.groupby('SelectionCohort').size().sort_index().to_string())
print('')
print(selected_samples_df[['SelectionCohort', 'DatasetIndex', 'PatientID', 'Cohort', 'Total Error', 'SelectionMargin']].to_string(index=False))

noise_levels = [0.00, 0.05, 0.10, 0.20, 0.30]
n_trials = 10
noise_methods = ['feature_noise', 'matrix_noise_rebuild', 'edge_perturbation', 'conditioning_noise']

records = []
for _, row in selected_samples_df.iterrows():
    ds_idx = int(row['DatasetIndex'])
    base_sample = test_dataset[ds_idx]
    patient_id = str(row['PatientID'])
    selection_cohort = str(row['SelectionCohort'])
    cohort_threshold = cohort_thresholds[selection_cohort]

    for method in noise_methods:
        for noise_level in noise_levels:
            for trial in range(n_trials):
                x_err, adj_err, total_err = infer_total_error(base_sample, noise_method=method, noise_level=float(noise_level))
                cohort_pred = cohort_positive_prediction(float(total_err), cohort_threshold)
                is_cohort_stable = int(cohort_pred == 1)
                records.append(
                    {
                        'SelectionCohort': selection_cohort,
                        'PatientID': patient_id,
                        'DatasetIndex': ds_idx,
                        'Method': method,
                        'NoiseLevel': float(noise_level),
                        'NoiseLevelPercent': float(noise_level * 100.0),
                        'Trial': trial,
                        'X Error': float(x_err),
                        'Adj Error': float(adj_err),
                        'Total Error': float(total_err),
                        'CohortPredPositive': cohort_pred,
                        'CohortStable': is_cohort_stable,
                    }
                )

robustness_df = pd.DataFrame(records)
summary_df = (
    robustness_df
    .groupby(['SelectionCohort', 'Method', 'NoiseLevel', 'NoiseLevelPercent'], as_index=False)
    .agg(
        MeanTotalError=('Total Error', 'mean'),
        StdTotalError=('Total Error', 'std'),
        CohortStabilityRate=('CohortStable', 'mean')
    )
)

print('')
print('Cohort stability summary (higher is better for every cohort):')
print(
    summary_df[['SelectionCohort', 'Method', 'NoiseLevelPercent', 'CohortStabilityRate']]
    .sort_values(['SelectionCohort', 'Method', 'NoiseLevelPercent'])
    .to_string(index=False)
)

for cohort_name in cohorts_to_analyze:
    cohort_summary = summary_df[summary_df['SelectionCohort'] == cohort_name]
    if cohort_summary.empty:
        continue

    cohort_threshold_error = cohort_thresholds[cohort_name]['threshold_error']

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for method in noise_methods:
        m = cohort_summary[cohort_summary['Method'] == method]
        axes[0].plot(m['NoiseLevelPercent'], m['MeanTotalError'], marker='o', label=method)
    axes[0].axhline(cohort_threshold_error, linestyle='--', color='black', alpha=0.6, label=f'{cohort_name} threshold')
    axes[0].set_xlabel('Noise level (%)')
    axes[0].set_ylabel('Mean total reconstruction error')
    axes[0].set_title(f'Error drift under noise ({cohort_name})')
    axes[0].legend()

    for method in noise_methods:
        m = cohort_summary[cohort_summary['Method'] == method]
        axes[1].plot(m['NoiseLevelPercent'], m['CohortStabilityRate'], marker='o', label=method)
    axes[1].set_xlabel('Noise level (%)')
    axes[1].set_ylabel('Cohort-correct prediction rate')
    axes[1].set_ylim(0, 1)
    axes[1].set_title(f'Decision stability under noise ({cohort_name})')
    axes[1].legend()

    plt.tight_layout()
    plt.show()

eval_base_dir = Path(run_artifact_dir) if 'run_artifact_dir' in globals() else Path.cwd()
eval_dir = eval_base_dir / f'robustness_eval_{datetime.now().strftime("%Y-%m-%d_%H-%M-%S")}'
eval_dir.mkdir(parents=True, exist_ok=True)

selected_file = eval_dir / 'selected_top5_by_cohort.csv'
summary_file = eval_dir / 'robustness_summary_by_cohort.csv'
details_file = eval_dir / 'robustness_details_by_cohort.csv'
thresholds_file = eval_dir / 'cohort_thresholds.csv'
meta_file = eval_dir / 'robustness_meta.json'

selected_samples_df.to_csv(selected_file, index=False)
summary_df.to_csv(summary_file, index=False)
robustness_df.to_csv(details_file, index=False)
pd.DataFrame(threshold_rows).to_csv(thresholds_file, index=False)

meta = {
    'threshold_type': 'one_vs_rest_per_cohort_from_validation_reconstruction_error',
    'cohort_thresholds': cohort_thresholds,
    'noise_levels': noise_levels,
    'noise_levels_percent': [float(x * 100.0) for x in noise_levels],
    'n_trials': n_trials,
    'methods': noise_methods,
    'cohorts_to_analyze': cohorts_to_analyze,
    'top_k_per_cohort': top_k,
    'selected_count_by_cohort': selected_samples_df.groupby('SelectionCohort').size().to_dict(),
}
with open(meta_file, 'w') as f:
    json.dump(meta, f, indent=2)

print(f'Saved evaluation artifacts to: {eval_dir}')